In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dsc592; 
USE CATALOG dsc592;
CREATE SCHEMA IF NOT EXISTS weather;

CREATE VOLUME IF NOT EXISTS weather.landing
  COMMENT 'Demo 1 landing zone: raw API responses';

LIST '/Volumes/dsc592/weather/landing/';

## read the csv with SQL

In [0]:
%sql
SELECT * 
FROM read_files('/Volumes/dsc592/weather/landing/cities.csv', format => 'csv', header => true)

## read the csv using PySpark

In [0]:
c = spark.read.csv('/Volumes/dsc592/weather/landing/cities.csv', header=True, inferSchema=True)
display(c)

# Write a file into the volume

In [0]:
import json, os, requests
from datetime import datetime, timezone

In [0]:
CATALOG = "dsc592" 

sends one HTTP GET request to the Open-Meteo weather API and stores the server’s reply in r

In [0]:
r = requests.get("https://api.open-meteo.com/v1/forecast", timeout=10,
                 params={"latitude": 34.23, "longitude": -77.94,
                         "current": "temperature_2m,wind_speed_10m",
                         "timezone": "UTC"})

`requests` builds the full URL from params, so the line is the same as opening this address:
`https://api.open-meteo.com/v1/forecast?latitude=34.23&longitude=-77.94&current=temperature_2m,wind_speed_10m&timezone=UTC`

`r` is a response object, not the data itself.

In [0]:
r.raise_for_status()     # if the server answered with an error (404, 429, 500...), stop with an exception
data = r.json()          # turn the JSON text into a Python dict
data["current"]          

In [0]:
data.keys()

### we will save `data` as wilmington.json
in `landing/test/2026/11/DD/HH`

In [0]:
now = datetime.now(timezone.utc)
out_dir = f"/Volumes/dsc592/weather/landing/test/{now:%Y/%m/%d/%H}"
out_dir

In [0]:
now.isoformat()

In [0]:
os.makedirs(out_dir, exist_ok=True)

- `json.dump(dict, f)`: Converts the Python dictionary `dict` to JSON text and writes it into `f`
- A datetime cannot go into JSON directly, so `.isoformat()` turns it into a standard string.
- `**data`: unpacks `data`

In [0]:
data = r.json() 
with open(f"{out_dir}/wilmington.json", "w") as f:
    json.dump({"city": "Wilmington", "fetched_at": now.isoformat(), **data}, f)